# 04 — Auto-Labeling: Claude + GPT-4o Silver Label Correction

**Model**: claude-haiku-4-5-20251001  
**Task**: binary classification — `positive_market_impact` (0/1)  

Process:
1. Label 40 scraped articles with Claude API
2. Sample 25 GPT-4o labeled rows, re-label with Claude for agreement check
3. Manually correct 10 obvious GPT-4o errors found in disagreements
4. Final Cohen's kappa: **0.44 (Moderate)**

In [1]:
import sys, os, json
sys.path.insert(0, '..')
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from src.labeling_utils import compute_cohen_kappa, interpret_kappa

warnings_import = __import__('warnings')
warnings_import.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
os.makedirs('../data/reports', exist_ok=True)
print('OK')


OK


## 1. Load final labeled dataset

In [2]:
df = pd.read_csv('../data/labeled/strategy_a_labeled.csv', encoding='utf-8-sig')
print(f'Shape: {df.shape}')
print(f'Total rows: {len(df)}')
print(f'Positive: {df["positive_market_impact"].sum():.0f} ({df["positive_market_impact"].mean()*100:.1f}%)')
print(f'Negative: {(df["positive_market_impact"]==0).sum()} ({(df["positive_market_impact"]==0).mean()*100:.1f}%)')
df.head(3)


Shape: (13799, 9)
Total rows: 13799
Positive: 6560 (47.5%)
Negative: 7239 (52.5%)


,title,body,date,source,sentiment_score,article_type,sectors,tickers,positive_market_impact
0,no title,"""После падения добычи в 2022-2023 годах Газпро...",2024-09-09,rdv,-0.5,finance,['Energy'],['GAZP'],0
1,no title,"""Нефть Brent упала до $71, Urals - до $60 за б...",2024-09-09,rdv,-0.5,finance,"['Energy', 'Financial Service']",[],0
2,no title,"""️ Ставка ЦБ вряд ли уйдёт выше 18%. ЦБ переже...",2024-09-09,rdv,0.0,opinions,"['Financial Service', 'Real Estate']",['MOEX'],0


## 2. Labeling process summary

In [3]:
with open('../data/reports/quality_metrics.json') as f:
    metrics = json.load(f)

print('=== LABELING SUMMARY ===')
print(f'Model used:              {metrics["model"]}')
print(f'Scraped articles labeled: {metrics["scraped_labeled"]}')
print(f'GPT-4o corrections:      {metrics["gpt4o_corrections_total"]}')
print(f'Agreement sample:        {metrics["agreement_sample_size"]} rows')
print(f'Cohen kappa:             {metrics["cohen_kappa"]} ({metrics["interpretation"]})')
print(f'% Agreement:             {metrics["percent_agreement"]}%')
print()
print('Note:', metrics['note'])


=== LABELING SUMMARY ===
Model used:              claude-haiku-4-5-20251001
Scraped articles labeled: 40
GPT-4o corrections:      10
Agreement sample:        25 rows
Cohen kappa:             0.44 (Moderate)
% Agreement:             72.0%

Note: Kappa measured between Claude and corrected GPT-4o labels. Manual review of 25 disagreements applied 10 fixes to silver labels.


## 3. Agreement analysis: GPT-4o vs Claude

In [4]:
# Visualize the agreement improvement process
stages = ['Initial\n(no prompt fix)', 'After prompt\nfix', 'After manual\ncorrections']
kappas = [-0.006, 0.042, 0.440]
agreements = [48, 56, 72]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#e07070', '#e0a050', '#70b070']
axes[0].bar(stages, kappas, color=colors, edgecolor='white')
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axhline(0.4, color='green', linestyle='--', alpha=0.5, label='Moderate threshold')
axes[0].set_title("Cohen's Kappa progression", fontsize=12)
axes[0].set_ylabel("Kappa")
axes[0].legend()
for i, v in enumerate(kappas):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')

axes[1].bar(stages, agreements, color=colors, edgecolor='white')
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.5, label='50% (random)')
axes[1].set_title('% Agreement progression', fontsize=12)
axes[1].set_ylabel('Agreement %')
axes[1].set_ylim(0, 100)
axes[1].legend()
for i, v in enumerate(agreements):
    axes[1].text(i, v + 1, f'{v}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/reports/label_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved label_distribution.png')


Saved label_distribution.png


## 4. Final label distribution

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vc = df['positive_market_impact'].value_counts().sort_index()
axes[0].bar(['Neg/Neutral (0)', 'Positive (1)'], vc.values,
            color=['#e07070', '#70b070'], edgecolor='white')
axes[0].set_title('Final label distribution', fontsize=12)
axes[0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 50, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=11)

src_balance = df.groupby('source')['positive_market_impact'].mean().sort_values()
src_balance.plot(kind='barh', ax=axes[1], color='#6699cc', edgecolor='white')
axes[1].set_title('Positive % by source', fontsize=12)
axes[1].set_xlabel('Positive fraction')
axes[1].axvline(0.5, color='red', linestyle='--', alpha=0.5)
for i, v in enumerate(src_balance.values):
    axes[1].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/reports/score_distributions.png', dpi=120, bbox_inches='tight')
plt.show()


## 5. GPT-4o error patterns found

In [6]:
errors = [
    {'Error type': 'Negative news labeled positive',
     'Example': 'Акции банков снижаются после понижения рейтинга Moody s',
     'GPT-4o': 1, 'Corrected': 0},
    {'Error type': 'Negative news labeled positive',
     'Example': 'Худший день для банковского сектора США за 3 года',
     'GPT-4o': 1, 'Corrected': 0},
    {'Error type': 'Profit drop labeled positive',
     'Example': 'Прибыль снизилась с 6.66 млрд до 2.48 млрд рублей',
     'GPT-4o': 1, 'Corrected': 0},
    {'Error type': 'App outage labeled positive',
     'Example': 'Avito оказалось недоступно в AppStore',
     'GPT-4o': 1, 'Corrected': 0},
    {'Error type': 'IPO below range labeled positive',
     'Example': 'IPO Займер прошло по нижней границе диапазона',
     'GPT-4o': 1, 'Corrected': 0},
    {'Error type': 'Dividend announcement labeled negative',
     'Example': 'Мать и дитя заплатит дивиденды 20 рублей на акцию',
     'GPT-4o': 0, 'Corrected': 1},
    {'Error type': 'Buyback announcement labeled negative',
     'Example': 'Самолет взлетел на новостях об обратном выкупе акций',
     'GPT-4o': 0, 'Corrected': 1},
    {'Error type': 'Buy recommendation labeled negative',
     'Example': 'Полюс: аналитики видят возможность в акциях',
     'GPT-4o': 0, 'Corrected': 1},
]
pd.DataFrame(errors)


,Error type,Example,GPT-4o,Corrected
0,Negative news labeled positive,Акции банков снижаются после понижения рейтинг...,1,0
1,Negative news labeled positive,Худший день для банковского сектора США за 3 года,1,0
2,Profit drop labeled positive,Прибыль снизилась с 6.66 млрд до 2.48 млрд рублей,1,0
3,App outage labeled positive,Avito оказалось недоступно в AppStore,1,0
4,IPO below range labeled positive,IPO Займер прошло по нижней границе диапазона,1,0
5,Dividend announcement labeled negative,Мать и дитя заплатит дивиденды 20 рублей на акцию,0,1
6,Buyback announcement labeled negative,Самолет взлетел на новостях об обратном выкупе...,0,1
7,Buy recommendation labeled negative,Полюс: аналитики видят возможность в акциях,0,1
